# ALMA-C11 + CEERS unified analogue selection and 5-arm powderday RT staging

Extends the CEERS-pair pipeline (`analogues_specphot_cis100_rt_agn.ipynb`) to the 18 ALMA-C11
sources in `ALMAC11_sed_ism_modeling_results.csv` (z = 0.338-0.427), unified with CEERS
719/2962 into one 20-target campaign. Per selected analogue up to five RT arms:

| run_tag | paramf | galaxies | purpose |
|---|---|---|---|
| `dust_on` | `parameters_master.py` | union (minus reusable) | fiducial AGN-off, dust on, 4 sightlines |
| `dust_off` | `parameters_master-nodust.py` | ALL union | dust-free stellar baseline -> attenuation |
| `nenkova_i90` | `parameters_master-nenkova-i90.py` | AGN hosts (minus reusable) | edge-on CLUMPY torus, 1 sightline |
| `nenkova_i60` | `parameters_master-nenkova-i60.py` | AGN hosts | intermediate torus view, 1 sightline |
| `nenkova_i30` | `parameters_master-nenkova-i30.py` | AGN hosts | near-face-on torus view, 1 sightline |

Output trees: `output/cis100/sed_almac11/<run_tag>/powderday_sed_out/snap_NNN/gal_G/`.
Selection tables + figures + maps: `output/cis100/almac11_specphot/`.

**Design decisions (locked 2026-08-04):**
- Nenkova inclination arms run only for AGN-host analogues, **single sightline** `THETA=[0]`
  (the CLUMPY `i` only reshapes the injected spectrum; 4-angle test agreed to <0.5%). Compare
  against sightline `findx=0` of the 4-angle dust arms.
- `MAX_PER_GROUP = 20` per (target, AGN class); union deduplicated on `(snap, gid)` before RT —
  a galaxy analogous to several targets is extracted and RT'd once (`MEMBERS` HDU keeps the map).
- **`dust_off` is never reused** from the old `nodust_chab_agn` tree: that run had `BH_SED=True`
  (Hopkins), so its dust-free SEDs contain AGN light and are not a clean stellar baseline.
- Reused from the CEERS campaign (snaps 091/096, verified with `sed_is_complete`): `dust_on` via
  `agnoff_source_map_chab.fits`, `nenkova_i90` via `dusty_simdust_chab_agn_nenkova` (4-sightline;
  analysis uses the sightline mean there).
- AGN gate `f_Edd >= 0.02 & L_bol >= 1e43`; if a target's AGN class has < 5 members the L_bol-only
  fallback gate is used for that target (recorded in `AGN_GATE`).

**Run order:** Parts 0-3 are cheap and read-only outside `almac11_specphot/` — run them, review
the counts and the cost table, tune `OVERRIDES`/caps, then Part 4 (heavy extraction), Part 5
(staging), Part 5c (verification). Jobs are written, **never submitted from here**.


## Part 0 · configuration

Targets are built from the CSV (`ID`, `z`, `log(Mstar/Msol)`, `Age(log/yr)`); the anchor snapshot
is the nearest-z snapshot **that has a caesar catalog**. The two CEERS entries are appended with
their Chabrier values and their historical windows (via `OVERRIDES`), so their selection
reproduces the chab run exactly.


In [ ]:
# ── Part 0 · configuration ───────────────────────────────────────────────────
import os, glob, sys, re, inspect
from collections import defaultdict

import numpy as np
import h5py
from astropy.table import Table
from astropy.io import fits
from astropy.cosmology import Planck13
import astropy.units as u
from astropy import constants as const

from simbanator.io.simba import Simulation
from simbanator.sed.makesed import MakeSED

sim = Simulation('cis100')

HOME            = '/mnt/home/glorenzon/analize_simba_cgm'
OUTDIR          = os.path.join(HOME, 'output', 'cis100', 'almac11_specphot')  # tables + figures
SED_OUT         = os.path.join(HOME, 'output', 'cis100', 'sed_almac11')       # RT trees, one per arm
SED_ANALOGUES   = os.path.join(HOME, 'output', 'cis100', 'sed_analogues')     # old CEERS trees (read-only)
CEERS_OUTDIR    = os.path.join(HOME, 'output', 'cis100', 'analogues_specphot')
PARTICLE_PREFIX = 'm100n1024'
hydro_dir_base  = os.path.join(HOME, 'output', sim.name, 'filtered_particles')  # shared cutouts
PARTITION       = 'INTEL_SKYLAKE,INTEL_CASCADE,INTEL_PHI,INTEL_HASWELL'
ALMAC11_CSV     = os.path.join(HOME, 'ALMAC11_sed_ism_modeling_results.csv')
os.makedirs(os.path.join(OUTDIR, 'figures'), exist_ok=True)

# ── the five arms: run_tag -> paramf + expected knobs (Part 1/5c assert these) ──
ARMS = {
    'dust_on':     dict(paramf='parameters_master.py',
                        expect=dict(BH_SED='False', dust_grid_type='manual', n_theta=4, incl=None),
                        who='union'),
    'dust_off':    dict(paramf='parameters_master-nodust.py',
                        expect=dict(BH_SED='False', dust_grid_type='dtm', n_theta=4, incl=None),
                        who='union'),
    'nenkova_i90': dict(paramf='parameters_master-nenkova-i90.py',
                        expect=dict(BH_SED='True', BH_model='Nenkova', dust_grid_type='manual',
                                    n_theta=1, incl=90),
                        who='agn'),
    'nenkova_i60': dict(paramf='parameters_master-nenkova-i60.py',
                        expect=dict(BH_SED='True', BH_model='Nenkova', dust_grid_type='manual',
                                    n_theta=1, incl=60),
                        who='agn'),
    'nenkova_i30': dict(paramf='parameters_master-nenkova-i30.py',
                        expect=dict(BH_SED='True', BH_model='Nenkova', dust_grid_type='manual',
                                    n_theta=1, incl=30),
                        who='agn'),
}

# ── selection constants (same physics as the CEERS chab run) ─────────────────
PASSIVE_FACTOR       = 0.2
NGAS_MIN, NSTAR_MIN  = 21, 20
EPS_RAD, AGN_FEDD_MIN, AGN_LBOL_MIN = 0.1, 0.02, 1e43
MAX_PER_GROUP        = 20    # cap per (target, AGN class), nearest kept
AGN_MIN_PER_TARGET   = 5     # below this, the L_bol-only fallback gate is used for that target
DLOGM_DEFAULT        = 0.30
DAGE_GYR_DEFAULT     = 1.00

# per-target window overrides (CEERS values reproduce the chab-run selection exactly)
OVERRIDES = {
    '719':  dict(dlogm=0.30, dage_gyr=0.85),
    '2962': dict(dlogm=0.40, dage_gyr=1.00),
}

# ── unified target table: 18 ALMA-C11 + 2 CEERS ──────────────────────────────
def _has_catalog(snap):
    try:
        return os.path.exists(sim.get_caesar_file(int(snap)))
    except Exception:
        return False

_z_all  = 1.0 / np.asarray(sim.scale_factors) - 1.0
_usable = np.array([s for s in range(len(_z_all)) if _has_catalog(s)], int)

def nearest_snap(z):
    j = int(np.argmin(np.abs(_z_all[_usable] - z)))
    return int(_usable[j]), float(_z_all[_usable[j]])

C = Table.read(ALMAC11_CSV, format='csv')
TARGETS = {}
for r in C:
    tid = str(r['ID']).strip()
    snap, z_snap = nearest_snap(float(r['z']))
    TARGETS[tid] = dict(z=float(r['z']), snap=snap, z_snap=z_snap,
                        logm=float(r['log(Mstar/Msol)']),
                        age_gyr=10.0 ** float(r['Age(log/yr)']) / 1e9,
                        sample='almac11')

# CEERS pair (Chabrier values, analogues_specphot_cis100_rt_agn.ipynb Part 0)
TARGETS['719']  = dict(z=1.463, snap=91, z_snap=1.4584, logm=10.29, age_gyr=2.65, sample='ceers')
TARGETS['2962'] = dict(z=1.266, snap=96, z_snap=1.2780, logm=10.68, age_gyr=3.30, sample='ceers')

for tid, T in TARGETS.items():
    ov = OVERRIDES.get(tid, {})
    T['dlogm']    = ov.get('dlogm', DLOGM_DEFAULT)
    T['dage_gyr'] = ov.get('dage_gyr', DAGE_GYR_DEFAULT)
    T['tid_safe'] = re.sub(r'[^A-Za-z0-9_-]', '', tid)

PREFLIGHT_OK = False   # Part 1 sets this; heavy cells refuse to run without it

print(f"{len(TARGETS)} targets "
      f"({sum(1 for T in TARGETS.values() if T['sample'] == 'almac11')} ALMA-C11 + "
      f"{sum(1 for T in TARGETS.values() if T['sample'] == 'ceers')} CEERS)\n")
print(f"{'tid':8s} {'z':>7s} {'snap':>4s} {'z_snap':>7s} {'|dz|':>6s} "
      f"{'logM*':>6s} {'age':>5s} {'±dm':>5s} {'±da':>5s}")
for tid, T in sorted(TARGETS.items(), key=lambda kv: kv[1]['z']):
    print(f"{tid:8s} {T['z']:7.4f} {T['snap']:4d} {T['z_snap']:7.4f} "
          f"{abs(T['z'] - T['z_snap']):6.4f} {T['logm']:6.2f} {T['age_gyr']:5.2f} "
          f"{T['dlogm']:5.2f} {T['dage_gyr']:5.2f}")

_snap_counts = defaultdict(int)
for T in TARGETS.values():
    _snap_counts[T['snap']] += 1
print('\nanchor snapshots (each NEW snap costs one ~265 GB extraction read):')
for s in sorted(_snap_counts):
    _have = os.path.isdir(os.path.join(hydro_dir_base, f'snap_{s:03d}'))
    print(f"  snap {s:03d} (z={sim.get_z_from_snap(s):.4f}): {_snap_counts[s]:2d} targets"
          + ('' if _have else '   <- NEW (extraction needed)'))


## Part 1 · preflight — will the five arms actually run what we think?

Read-only. Checks, in order: (1a-1c) the powderday tree the jobs import, with the BH field-name
fix in place; (2) each arm's parameter master exists **in the imported simbanator copy**
(`paramf` is resolved there, `makesed.py:325`) and carries the expected knobs; (3) the CLUMPY
grid the Nenkova arms need; (4) arm homogeneity — every master may differ from `dust_on` only in
BH/torus/sightline/dust-off knobs, else the arm differences stop being interpretable;
(5) catalogs and snapshots exist for every anchor snap.


In [ ]:
# ── Part 1 · PREFLIGHT (read-only; no powderday import) ──────────────────────
import importlib.util as _ilu, zipfile as _zf, contextlib, io

_fail, _warn = [], []
_SEDDIR = os.path.dirname(inspect.getfile(MakeSED))
print('paramf files resolve against the IMPORTED simbanator:', _SEDDIR)
_repo_sed = os.path.join(HOME, 'simbanator', 'sed')
if os.path.realpath(_SEDDIR) != os.path.realpath(_repo_sed):
    _warn.append(f'imported simbanator ({_SEDDIR}) is not the repo copy ({_repo_sed}); '
                 'sync/reinstall first or the new paramf files may be missing/stale there.')

# 1a. the powderday copy the JOBS will import ---------------------------------
_setup = os.path.join(_SEDDIR, 'cosmology_setup_all_cluster.cis.sh')
_m = re.search(r'PD_FRONT_END=\\?"([^"\\]+)', open(_setup).read()) if os.path.exists(_setup) else None
if _m is None:
    _fail.append(f'could not read PD_FRONT_END out of {_setup}')
    PD_FRONT_END = PD_DIR = None
else:
    PD_FRONT_END = _m.group(1)
    PD_DIR = os.path.dirname(PD_FRONT_END)
    print(f'jobs run       : python {PD_FRONT_END} . parameters_master snapNN_ID')
    print(f'=> sys.path[0] : {PD_DIR}   (wins over the installed egg)')
    if not os.path.exists(PD_FRONT_END):
        _fail.append(f'{PD_FRONT_END} does not exist')

# 1b. BH field names in gadget2pd.py (malformed names crash EVERY BH_SED job) --
_BAD_MARKER = '("bh' + "','"
_WANT = ['coordinates', 'luminosity', 'nu', 'sed']
if PD_DIR:
    _g2p = os.path.join(PD_DIR, 'powderday', 'front_ends', 'gadget2pd.py')
    if not os.path.exists(_g2p):
        _fail.append(f'{_g2p} not found')
    else:
        _src = open(_g2p).read()
        _bad = _src.count(_BAD_MARKER)
        _good = sorted(set(re.findall(r'add_field\(\("bh","(\w+)"\)', _src)))
        print(f'BH field names : malformed {_bad} | correct {_good}')
        if _bad or _good != _WANT:
            _fail.append('gadget2pd.py BH field names are malformed — every BH_SED=True job '
                         'would raise ValueError. Re-apply the fix (*.pre-bhfix-20260730.bak).')
    _sc = os.path.join(PD_DIR, 'powderday', 'source_creation.py')
    if os.path.exists(_sc) and 'reg["bh","sed"]' not in open(_sc).read().replace("'", '"'):
        _fail.append('source_creation.py no longer indexes reg["bh","sed"] — field-name '
                     'contract changed; re-derive expectations before trusting 1b.')

# 2. per-arm parameter masters ------------------------------------------------
def _parse_master(path):
    src = open(path).read()
    v = {}
    for key in ('BH_SED', 'BH_var', 'dust_grid_type', 'BH_model',
                'dusttometals_ratio', 'n_photons_raytracing_dust'):
        m = re.search(rf'(?m)^{key}\s*=\s*(.+?)\s*(#.*)?$', src)
        v[key] = m.group(1).strip().strip('"\'') if m else None
    m = re.search(r'(?m)^nenkova_params\s*=\s*\[([^\]]+)\]', src)
    v['incl'] = int(float(m.group(1).split(',')[2])) if m else None
    m = re.search(r'(?m)^THETA\s*=\s*\[([^\]]+)\]', src)
    v['n_theta'] = len(m.group(1).split(',')) if m else None
    return v, src

print('\nper-arm parameter masters:')
_master_src = {}
for arm, spec in ARMS.items():
    p = os.path.join(_SEDDIR, spec['paramf'])
    if not os.path.exists(p):
        _fail.append(f"{arm}: {spec['paramf']} not found in {_SEDDIR}")
        continue
    v, _master_src[arm] = _parse_master(p)
    exp = spec['expect']
    bad = [k for k, want in exp.items() if want is not None and v.get(k) != str(want)
           and v.get(k) != want]
    print(f"  {arm:12s} BH_SED={v['BH_SED']:5s} model={str(v['BH_model']):8s} "
          f"i={str(v['incl']):4s} n_theta={v['n_theta']} grid={v['dust_grid_type']} "
          f"-> {'PASS' if not bad else 'FAIL ' + str(bad)}")
    if bad:
        _fail.append(f"{arm}: {spec['paramf']} knobs {bad} do not match {exp}")
    if v['BH_SED'] == 'True' and v['BH_var'] != 'False':
        _fail.append(f'{arm}: BH_var must be False to keep L_AGN consistent with the f_Edd split')

# 3. CLUMPY grid needed by the Nenkova arms -----------------------------------
_clumpy = os.path.join(os.environ.get('POWDERDAY_ROOT', os.path.expanduser('~')),
                       'powderday', 'agn_models', 'clumpy_models_201410_tvavg.hdf5')
if os.path.exists(_clumpy):
    print(f'\nCLUMPY grid    : {_clumpy}  ({os.path.getsize(_clumpy) / 1e9:.2f} GB)')
else:
    _fail.append(f'CLUMPY grid missing: {_clumpy} — the three Nenkova arms cannot run.')

# 4. arm homogeneity: only BH/torus/sightline/dust-off knobs may differ -------
_ALLOWED = {'BH_SED', 'BH_eta', 'BH_model', 'BH_modelfile', 'BH_var', 'nenkova_params',
            'THETA', 'PHI', 'dust_grid_type', 'dusttometals_ratio', 'n_photons_raytracing_dust'}
_ref = [l for l in _master_src.get('dust_on', '').splitlines()
        if l.strip() and not l.strip().startswith('#')]
for arm, src in _master_src.items():
    if arm == 'dust_on':
        continue
    _b = [l for l in src.splitlines() if l.strip() and not l.strip().startswith('#')]
    _keys = set()
    for l in sorted(set(_ref) ^ set(_b)):
        mk = re.match(r'\s*(\w+)\s*=', l)
        _keys.add(mk.group(1) if mk else l.strip())
    _extra = sorted(_keys - _ALLOWED)
    print(f'  {arm:12s} differs from dust_on in: {sorted(_keys)}')
    if _extra:
        _fail.append(f'{arm}: differs from dust_on outside the allowed knobs ({_extra}) — '
                     'arm differences would no longer isolate torus/dust effects.')

# 5. catalogs + snapshots for every anchor snap -------------------------------
print()
for s in sorted({T['snap'] for T in TARGETS.values()}):
    _cat = os.path.exists(sim.get_caesar_file(s))
    _snp = os.path.exists(sim.get_snapshot_file(s))
    _cut = os.path.isdir(os.path.join(hydro_dir_base, f'snap_{s:03d}'))
    print(f'  snap {s:03d}: catalog {_cat} | snapshot {_snp} | cutout dir {_cut}')
    if not _cat:
        _fail.append(f'snap {s}: no caesar catalog')
    if not _snp and not _cut:
        _fail.append(f'snap {s}: neither snapshot (for extraction) nor existing cutouts')

print('\n' + '=' * 78)
for w in _warn:
    print('WARN:', w)
if _fail:
    for f_ in _fail:
        print('FAIL:', f_)
    raise RuntimeError('Preflight failed — do NOT run the extraction or staging cells.')
PREFLIGHT_OK = True
print('PREFLIGHT PASSED — all five arms are wired up.')


## Part 2 · selection — 20 targets, cached catalog reads, AGN-thin fallback

Same physics as the CEERS chab run: box windows in log M* and mass-weighted age,
`sSFR < 0.2/t_H(z_snap)`, resolution floors, AGN gate `f_Edd >= 0.02 & L_bol >= 1e43`
(`L_bol = 0.1 Mdot c^2`). New here: caesar arrays are cached per snapshot (many targets share
one), and a target whose AGN class has fewer than `AGN_MIN_PER_TARGET` members falls back to the
L_bol-only gate for its AGN class (recorded per target in `AGN_GATE`; both `IS_AGN` and
`IS_AGN_LBOL` columns are always carried).

If a target's age window is empty (CIGALE ages close to t_H have no simulated twins —
e.g. hc6 at 9.0 Gyr), the window auto-widens in 0.5 Gyr steps (up to ±3.5 Gyr) until
`MIN_ANALOGUES` pass, keeping the oldest available quiescent analogues; every widening is
reported and the used half-width is stored as `dage_gyr_used`. Targets still empty after
widening are skipped and listed.

Products: `rt_selection_<tid>.fits` per target, `rt_union.fits` with HDUs `UNION`
(deduplicated galaxies + arm plan) and `MEMBERS` (target <-> galaxy many-to-many map), and a
selection-plane figure per snapshot.


In [ ]:
# ── Part 2 · selection ───────────────────────────────────────────────────────
import matplotlib
import matplotlib.pyplot as plt

_cat_cache = {}
def load_caesar_arrays(snap):
    if snap not in _cat_cache:
        with h5py.File(sim.get_caesar_file(int(snap)), 'r') as f:
            d = f['galaxy_data']
            c = dict(gid=np.asarray(d['GroupID'][:], int),
                     ms=np.asarray(d['dicts/masses.stellar'][:], float),
                     age=np.asarray(d['dicts/ages.mass_weighted'][:], float),
                     md=np.asarray(d['dicts/masses.dust'][:], float),
                     mgas=np.asarray(d['dicts/masses.gas'][:], float),
                     sfr=np.asarray(d['sfr'][:], float),
                     ngas=np.asarray(d['ngas'][:], int),
                     nstar=np.asarray(d['nstar'][:], int),
                     cen=np.asarray(d['central'][:], int),
                     fedd=np.asarray(d['bh_fedd'][:], float),
                     mdot=np.asarray(d['bhmdot'][:], float),
                     r50s=np.asarray(d['dicts/radii.stellar_half_mass'][:], float))
        with np.errstate(invalid='ignore', divide='ignore'):
            c['lm'] = np.log10(np.where(c['ms'] > 0, c['ms'], np.nan))
            c['ssfr'] = c['sfr'] / c['ms']
            c['lbol'] = (EPS_RAD * (c['mdot'] * u.Msun / u.yr) * const.c**2).to(u.erg / u.s).value
        c['agn'] = (np.nan_to_num(c['fedd']) >= AGN_FEDD_MIN) & \
                   (np.nan_to_num(c['lbol']) >= AGN_LBOL_MIN)
        c['agn_lbol'] = np.nan_to_num(c['lbol']) >= AGN_LBOL_MIN
        _cat_cache[snap] = c
    return _cat_cache[snap]

MIN_ANALOGUES   = 5      # auto-widen the age window until at least this many pass
DAGE_WIDEN_STEP = 0.5    # Gyr per widening step
DAGE_MAX        = 3.5    # Gyr, hard stop for the widening

SELECTED, EMPTY_TARGETS = {}, []
for tid, T in TARGETS.items():
    c = load_caesar_arrays(T['snap'])
    tH = Planck13.age(T['z_snap']).to(u.yr).value
    m_m = np.abs(c['lm'] - T['logm']) <= T['dlogm']
    m_q = c['ssfr'] < PASSIVE_FACTOR / tH
    m_r = (c['ngas'] >= NGAS_MIN) & (c['nstar'] >= NSTAR_MIN)

    # age window, auto-widened when the target is older than the SIMBA quiescent
    # population at this snapshot (CIGALE ages ~ t_H have no simulated twins: the
    # widening then keeps the OLDEST available quiescent analogues)
    dage = T['dage_gyr']
    while True:
        m_a = np.abs(c['age'] - T['age_gyr']) <= dage
        sel = m_m & m_a & m_q & m_r
        if int(sel.sum()) >= MIN_ANALOGUES or dage >= DAGE_MAX:
            break
        dage = min(dage + DAGE_WIDEN_STEP, DAGE_MAX)
    T['dage_gyr_used'] = dage
    if dage != T['dage_gyr']:
        _oldest = np.nanmax(np.where(m_q & m_r, c['age'], np.nan))
        print(f'  !! {tid}: age window widened ±{T["dage_gyr"]:.2f} -> ±{dage:.2f} Gyr '
              f'(target age {T["age_gyr"]:.2f} Gyr, t_H={tH / 1e9:.2f} Gyr, oldest '
              f'quiescent+resolved analogue {_oldest:.2f} Gyr)')

    n_agn_strict = int((sel & c['agn']).sum())
    if n_agn_strict >= AGN_MIN_PER_TARGET:
        agn_cls, gate = c['agn'], 'fedd+lbol'
    else:
        agn_cls, gate = c['agn_lbol'], 'lbol-only'
        print(f'  !! {tid}: only {n_agn_strict} strict-AGN in window '
              f'-> falling back to the L_bol-only gate')
    T['agn_gate'] = gate

    print(f"── {tid}  snap {T['snap']} (z={T['z_snap']:.4f})  logM*={T['logm']:.2f}"
          f"±{T['dlogm']:.2f}, age={T['age_gyr']:.2f}±{dage:.2f} Gyr ──")
    print(f"  mass {int(m_m.sum()):5d} | +age {int((m_m & m_a).sum()):5d} | "
          f"+quiescent {int((m_m & m_a & m_q).sum()):5d} | +resolution {int(sel.sum()):5d}"
          f"  -> AGN[{gate}] {int((sel & agn_cls).sum())} | non-AGN {int((sel & ~agn_cls).sum())}")

    dist = np.sqrt(((c['lm'] - T['logm']) / T['dlogm'])**2 +
                   ((c['age'] - T['age_gyr']) / dage)**2)
    keep = np.zeros(sel.size, bool)
    for cls_mask in (sel & agn_cls, sel & ~agn_cls):
        ii = np.where(cls_mask)[0]
        if MAX_PER_GROUP is not None and ii.size > MAX_PER_GROUP:
            ii = ii[np.argsort(dist[ii])[:MAX_PER_GROUP]]
        keep[ii] = True
    print(f"  staged (cap {MAX_PER_GROUP}/class): {int(keep.sum())}  -> "
          f"AGN {int((keep & agn_cls).sum())} | non-AGN {int((keep & ~agn_cls).sum())}")

    n = int(keep.sum())
    if n == 0:
        print(f'  !! {tid}: NO analogues even at ±{dage:.2f} Gyr — target SKIPPED\n')
        EMPTY_TARGETS.append(tid)
        continue
    A = Table(dict(SNAPSHOT=np.full(n, T['snap']), GROUPID_SNAPSHOT=c['gid'][keep],
                   REDSHIFT=np.full(n, T['z_snap']),
                   LOG_MSTAR=c['lm'][keep], AGE_STAR=c['age'][keep], SSFR=c['ssfr'][keep],
                   MGAS=c['mgas'][keep], MDUST=c['md'][keep],
                   LOG_MDUST=np.log10(np.where(c['md'][keep] > 0, c['md'][keep], np.nan)),
                   NGAS=c['ngas'][keep], NSTAR=c['nstar'][keep], CENTRAL=c['cen'][keep],
                   FEDD=c['fedd'][keep], LBOL=c['lbol'][keep],
                   IS_AGN=c['agn'][keep], IS_AGN_LBOL=c['agn_lbol'][keep],
                   R50_STAR=c['r50s'][keep], DIST_MATCH=dist[keep]))
    A['AGN_CLASS'] = np.where(agn_cls[keep], 'AGN', 'no_AGN')   # gate-dependent class
    A['AGN_GATE'] = np.full(n, gate)
    A['TARGET_ID'] = np.full(n, tid)
    A.sort('GROUPID_SNAPSHOT')
    A.write(os.path.join(OUTDIR, f"rt_selection_{T['tid_safe']}.fits"), overwrite=True)
    SELECTED[tid] = A
if EMPTY_TARGETS:
    print(f'targets with NO analogues (excluded from union/RT): {EMPTY_TARGETS}')
print()

# ── union (dedup on (snap, gid)) + many-to-many membership map ───────────────
union, members = {}, []
for tid, A in SELECTED.items():
    for r in A:
        key = (int(r['SNAPSHOT']), int(r['GROUPID_SNAPSHOT']))
        members.append((tid, key[0], key[1], str(r['AGN_CLASS']), float(r['DIST_MATCH'])))
        if key not in union:
            union[key] = dict(row=dict(zip(r.colnames, r)), agn_any=False)
        union[key]['agn_any'] |= (str(r['AGN_CLASS']) == 'AGN')

U = Table(rows=[(k[0], k[1], float(u_['row']['LOG_MSTAR']), float(u_['row']['AGE_STAR']),
                 float(u_['row']['SSFR']), float(u_['row']['FEDD']), float(u_['row']['LBOL']),
                 bool(u_['row']['IS_AGN']), bool(u_['row']['IS_AGN_LBOL']), bool(u_['agn_any']),
                 True, True, u_['agn_any'], u_['agn_any'], u_['agn_any'])
                for k, u_ in sorted(union.items())],
          names=('SNAPSHOT', 'GROUPID_SNAPSHOT', 'LOG_MSTAR', 'AGE_STAR', 'SSFR', 'FEDD', 'LBOL',
                 'IS_AGN', 'IS_AGN_LBOL', 'AGN_ANY',
                 'RUN_DUST_ON', 'RUN_DUST_OFF', 'RUN_I90', 'RUN_I60', 'RUN_I30'))
M = Table(rows=members, names=('TARGET_ID', 'SNAPSHOT', 'GROUPID_SNAPSHOT', 'AGN_CLASS',
                               'DIST_MATCH'))
_hdul = fits.HDUList([fits.PrimaryHDU(),
                      fits.table_to_hdu(U), fits.table_to_hdu(M)])
_hdul[1].name, _hdul[2].name = 'UNION', 'MEMBERS'
_hdul.writeto(os.path.join(OUTDIR, 'rt_union.fits'), overwrite=True)

print(f'union: {len(U)} unique galaxies over snaps '
      f'{sorted(set(int(s) for s in U["SNAPSHOT"]))} '
      f'({len(M)} member entries before dedup)')
print(f'  AGN hosts (torus arms): {int(np.asarray(U["AGN_ANY"]).sum())}')
for s in sorted(set(int(x) for x in U['SNAPSHOT'])):
    m_ = np.asarray(U['SNAPSHOT']) == s
    print(f'  snap {s:03d}: {int(m_.sum()):4d} unique | AGN {int((m_ & np.asarray(U["AGN_ANY"])).sum()):3d}')

# ── selection-plane figure per snapshot ──────────────────────────────────────
for s in sorted(set(int(x) for x in U['SNAPSHOT'])):
    c = load_caesar_arrays(s)
    tH = Planck13.age(sim.get_z_from_snap(s)).to(u.yr).value
    qui = (c['ssfr'] < PASSIVE_FACTOR / tH) & (c['ngas'] >= NGAS_MIN) & (c['nstar'] >= NSTAR_MIN)
    fig, ax = plt.subplots(figsize=(7, 5.5))
    ax.plot(c['lm'][qui], c['age'][qui], '.', ms=2, color='0.7', label='quiescent + resolved')
    m_ = np.asarray(U['SNAPSHOT']) == s
    a_ = m_ & np.asarray(U['AGN_ANY'])
    ax.plot(np.asarray(U['LOG_MSTAR'])[m_ & ~np.asarray(U['AGN_ANY'])],
            np.asarray(U['AGE_STAR'])[m_ & ~np.asarray(U['AGN_ANY'])], 'o', ms=4,
            color='tab:blue', mec='none', label='staged non-AGN')
    ax.plot(np.asarray(U['LOG_MSTAR'])[a_], np.asarray(U['AGE_STAR'])[a_], 'o', ms=4,
            color='tab:orange', mec='none', label='staged AGN host')
    for tid, T in TARGETS.items():
        if T['snap'] != s:
            continue
        ax.add_patch(plt.Rectangle((T['logm'] - T['dlogm'], T['age_gyr'] - T['dage_gyr']),
                                   2 * T['dlogm'], 2 * T['dage_gyr'],
                                   fill=False, ec='k', lw=0.8, alpha=0.6))
        ax.annotate(tid, (T['logm'], T['age_gyr'] + T['dage_gyr']), fontsize=7,
                    ha='center', va='bottom')
    ax.set(xlabel=r'$\log\,M_*\,[M_\odot]$', ylabel='mass-weighted age [Gyr]',
           title=f'snap {s:03d} (z={sim.get_z_from_snap(s):.4f}) — windows + staged analogues')
    ax.legend(fontsize=8, frameon=False)
    fig.tight_layout()
    fig.savefig(os.path.join(OUTDIR, 'figures', f'selection_plane_snap{s:03d}.png'),
                dpi=180, bbox_inches='tight')
    plt.show()


## Part 3 · reconcile — reuse finished CEERS products where physically valid

For union galaxies at snaps 091/096: `dust_on` resolves through `agnoff_source_map_chab.fits`
(trees `dusty_simdust` / `dusty_simdust_chab_topup`, both `BH_SED=False`), `nenkova_i90` through
`dusty_simdust_chab_agn_nenkova` — each only counts if `sed_is_complete()` passes (a
`.rtout.sed` on disk is NOT proof the RT finished; hyperion writes the `Peeled` SED groups
last). `dust_off` is never reused. Products: `reuse_map.fits` + the per-arm to-run lists `RUN`
and a cost table.


In [ ]:
# ── Part 3 · reconcile vs existing trees ─────────────────────────────────────
def rtout_path_tag(tag, snap, gid, root=None):
    return os.path.join(root or SED_OUT, tag, 'powderday_sed_out', f'snap_{int(snap):03d}',
                        f'gal_{int(gid)}', f'snap{int(snap):03d}.galaxy{int(gid):06d}.rtout.sed')

def sed_is_complete(path):
    # True only if the RT actually FINISHED (Peeled SED groups are written last).
    if not (path and os.path.exists(path)):
        return False
    try:
        with h5py.File(path, 'r') as f:
            g = f.get('Peeled')
            return bool(g) and any('seds' in g[k] for k in g.keys())
    except OSError:
        return False

_agnoff_map = {}
_agnoff_f = os.path.join(CEERS_OUTDIR, 'agnoff_source_map_chab.fits')
if os.path.exists(_agnoff_f):
    for r in Table.read(_agnoff_f):
        _agnoff_map[(int(r['SNAPSHOT']), int(r['GROUPID_SNAPSHOT']))] = str(r['AGNOFF_TAG'])

REUSE_ROWS = []                       # (arm, snap, gid, tree_tag, path, complete)
RUN = {arm: [] for arm in ARMS}
for key, u_ in sorted(union.items()):
    snap, gid = key
    # dust_on: reuse the CEERS AGN-off SED if the map has it and the RT finished
    reused = False
    if key in _agnoff_map:
        tag = _agnoff_map[key]
        p = rtout_path_tag(tag, snap, gid, root=SED_ANALOGUES)
        ok = sed_is_complete(p)
        REUSE_ROWS.append(('dust_on', snap, gid, tag, p, ok))
        reused = ok
    if not reused:
        RUN['dust_on'].append(key)
    # dust_off: NEVER reused (nodust_chab_agn had BH_SED=True -> AGN light in the baseline)
    RUN['dust_off'].append(key)
    # torus arms: AGN hosts only
    if u_['agn_any']:
        p = rtout_path_tag('dusty_simdust_chab_agn_nenkova', snap, gid, root=SED_ANALOGUES)
        if sed_is_complete(p):
            REUSE_ROWS.append(('nenkova_i90', snap, gid, 'dusty_simdust_chab_agn_nenkova', p, True))
        else:
            RUN['nenkova_i90'].append(key)
        RUN['nenkova_i60'].append(key)
        RUN['nenkova_i30'].append(key)

R = Table(rows=REUSE_ROWS or None,
          names=('ARM', 'SNAPSHOT', 'GROUPID_SNAPSHOT', 'TREE_TAG', 'RTOUT_PATH', 'COMPLETE'))
R.write(os.path.join(OUTDIR, 'reuse_map.fits'), overwrite=True)

print(f'{"arm":12s} {"to run":>7s} {"reused":>7s}   note')
_n_reuse = lambda a: sum(1 for r_ in REUSE_ROWS if r_[0] == a and r_[5])
for arm in ARMS:
    note = {'dust_off': 'no reuse by design (old nodust tree has BH_SED=True)'}.get(arm, '')
    print(f'{arm:12s} {len(RUN[arm]):7d} {_n_reuse(arm):7d}   {note}')

# rough cost, in units of one 4-sightline AGN-off (dust_on) galaxy:
#   dust_on 1.0 | dust_off ~0.2 (dust RT off) | nenkova ~2.0 (8x AGN cost / 4 for 1 sightline)
_units = (len(RUN['dust_on']) * 1.0 + len(RUN['dust_off']) * 0.2 +
          sum(len(RUN[a]) for a in ('nenkova_i90', 'nenkova_i60', 'nenkova_i30')) * 2.0)
print(f'\nestimated cost ~ {_units:.0f} dust_on-equivalent galaxy runs '
      f'(CEERS benchmark: ~9 cpu-h each at 1e6 photons)')
print('review this BEFORE Part 4/5; tighten MAX_PER_GROUP or the AGN cap if too heavy.')


## Part 4 · particle cutouts (**heavy** — reads each new ~265 GB snapshot once)

One `extract_particles` call per snapshot, only for galaxies whose cutout is missing or lacks
`PartType5` (needed by the torus arms; harmless under `BH_SED=False`). Existing complete cutouts
(all of snaps 091/096 after the 2026-07-30 re-extraction) are skipped.


In [ ]:
# ── Part 4 · extraction ──────────────────────────────────────────────────────
if not PREFLIGHT_OK:
    raise RuntimeError('Preflight did not pass — fix that before spending the snapshot reads.')

from simbanator.analysis import extract_particles

EXTRACT_PTYPES = ("PartType0", "PartType4", "PartType5")

def _cutout(snap, gid):
    return os.path.join(hydro_dir_base, f'snap_{int(snap):03d}',
                        f'{PARTICLE_PREFIX}_snap{int(snap):03d}_gal{int(gid):06d}.h5')

need = defaultdict(list)
for (snap, gid) in sorted(union):
    p = _cutout(snap, gid)
    ok = False
    if os.path.exists(p):
        try:
            with h5py.File(p, 'r') as f:
                ok = 'PartType5' in f    # BH data present -> cutout usable by every arm
        except OSError:
            ok = False
    if not ok:
        need[int(snap)].append(int(gid))

if not need:
    print('all cutouts already present with PartType5 — nothing to extract')
for snap in sorted(need):
    ids_here = np.unique(need[snap])
    simfile = sim.get_snapshot_file(snap)
    print(f'snap {snap:03d}: extracting {len(ids_here)} galaxies from {os.path.basename(simfile)}')
    cs = sim.load_catalog(snap=snap)
    extract_particles(cs, simfile, snap, galaxy_ids=ids_here, ptypes=EXTRACT_PTYPES,
                      sim_name=sim.name, prefix=PARTICLE_PREFIX, overwrite=True, verbose=1)
    del cs
print('\nextraction complete ->', hydro_dir_base)


In [ ]:
# ── Part 4b · verify PartType5 against caesar nbh (cheap; do not skip) ───────
_nbh_caesar = {}
for s in sorted(set(k[0] for k in union)):
    with h5py.File(sim.get_caesar_file(s), 'r') as f:
        _nbh_caesar[s] = dict(zip(np.asarray(f['galaxy_data/GroupID'][:], int),
                                  np.asarray(f['galaxy_data/nbh'][:], int)))

n_tot = n_with_bh = 0
missing, mismatch, agn_no_bh = [], [], []
for (snap, gid), u_ in sorted(union.items()):
    p = _cutout(snap, gid)
    if not os.path.exists(p):
        missing.append((snap, gid))
        continue
    n_tot += 1
    with h5py.File(p, 'r') as f:
        n_here = (f['PartType5/BH_Mass'].shape[0]
                  if 'PartType5' in f and 'BH_Mass' in f['PartType5'] else 0)
    if n_here:
        n_with_bh += 1
    if n_here != _nbh_caesar[snap].get(gid, 0):
        mismatch.append((snap, gid, n_here, _nbh_caesar[snap].get(gid, 0)))
    if u_['agn_any'] and n_here == 0:
        agn_no_bh.append((snap, gid))

print(f'cutouts found        : {n_tot} / {len(union)}')
print(f'  with >=1 black hole: {n_with_bh}')
if missing:
    print(f'  !! MISSING cutouts : {len(missing)} e.g. {missing[:5]}')
if mismatch:
    print(f'  !! nbh != caesar   : {len(mismatch)} e.g. {mismatch[:5]}')
if agn_no_bh:
    print(f'  !! AGN host without BH particle: {len(agn_no_bh)} e.g. {agn_no_bh[:5]}')
if missing or mismatch or agn_no_bh:
    raise RuntimeError('cutouts inconsistent with caesar — resolve before staging')
print('PartType5 verified — safe to stage.')


## Part 5 · stage the five arms (jobs written, **not submitted**)

One `MakeSED` per arm; each arm gets its own `selection_file` so the `target_selection/*.h5`
files never collide. Then Part 5c parses every staged `snap_*/parameters_master.py` and asserts
the knob tuple per tree before anything is submitted.


In [ ]:
# ── Part 5 · stage each arm ──────────────────────────────────────────────────
if not PREFLIGHT_OK:
    raise RuntimeError('Preflight did not pass.')

for arm, spec in ARMS.items():
    keys = RUN[arm]
    if not keys:
        print(f'{arm:12s}: nothing to run (all reused) — skipped')
        continue
    snaps = np.array([k[0] for k in keys], int)
    ids = np.array([k[1] for k in keys], int)
    ms = MakeSED(sim, nnodes=1, model_run_name=arm, hydro_dir_base=hydro_dir_base,
                 selection_file=f'rt_almac11_{arm}', output_dir=SED_OUT, run_tag=arm)
    ms.selection_gals(snaps=snaps, galaxyID=ids)
    ms.create_master('cluster', 'plist', radius=None, partition=PARTITION,
                     prefix=PARTICLE_PREFIX, paramf=spec['paramf'], snaps_to_run=None)
    print(f'{arm:12s}: staged {len(keys)} galaxies over snaps '
          f'{sorted(set(snaps.tolist()))} -> {os.path.join(SED_OUT, arm)}')


In [ ]:
# ── Part 5c · verify the staged parameter files, then print submit commands ──
_all_ok = True
for arm, spec in ARMS.items():
    if not RUN[arm]:
        continue
    exp = spec['expect']
    print(f'=== {arm} (expect BH_SED={exp["BH_SED"]}, i={exp["incl"]}, '
          f'n_theta={exp["n_theta"]}, grid={exp["dust_grid_type"]}) ===')
    for s in sorted(set(k[0] for k in RUN[arm])):
        jdir = os.path.join(SED_OUT, arm, 'powderday_sed_out', f'snap_{s:03d}')
        if not os.path.isdir(jdir):
            jdir = os.path.join(SED_OUT, arm, 'powderday_sed_out', f'snap_{s}')
        pm = os.path.join(jdir, 'parameters_master.py')
        if not os.path.exists(pm):
            print(f'  snap {s}: !! no parameters_master.py'); _all_ok = False; continue
        v, _ = _parse_master(pm)
        nids = (sum(1 for _ in open(os.path.join(jdir, 'ids.txt')))
                if os.path.exists(os.path.join(jdir, 'ids.txt')) else 0)
        nexp = sum(1 for k in RUN[arm] if k[0] == s)
        bad = [k for k, want in exp.items() if want is not None
               and v.get(k) != str(want) and v.get(k) != want]
        print(f'  snap {s:03d}: BH_SED={v["BH_SED"]} model={v["BH_model"]} i={v["incl"]} '
              f'n_theta={v["n_theta"]} grid={v["dust_grid_type"]} | ids {nids} (expect {nexp}) '
              f'-> {"OK" if not bad and nids == nexp else "FAIL"}')
        if bad or nids != nexp:
            _all_ok = False

print('\n' + ('staging verified' if _all_ok else '!! STAGING PROBLEM — do not submit'))
print('\n── submit with (cheap arms first, torus arms last; calibrate ONE galaxy first!) ──')
for arm in ('dust_off', 'dust_on', 'nenkova_i90', 'nenkova_i60', 'nenkova_i30'):
    for s in sorted(set(k[0] for k in RUN[arm])):
        jdir = os.path.join(SED_OUT, arm, 'powderday_sed_out', f'snap_{s:03d}')
        jobs = sorted(glob.glob(os.path.join(jdir, 'master.snap*.job')))
        if jobs:
            print(f'cd {jdir} && sbatch {os.path.basename(jobs[-1])}')
        elif RUN[arm]:
            print(f'!! no master.snap*.job in {jdir} — staging failed?')


## Part 6 · after the RT — census and the arm source map

**Calibration first**: before mass submission, run ONE AGN-host galaxy through all five arms
(edit its `master.snap*.job` plist to a single id, or submit only its `snapNNN_<gid>.py`) and
check: `dust_off >= dust_on` at every wavelength; Nenkova arms >= `dust_on` in the MIR;
UV suppressed for i=90 relative to i=30.

The census below needs `RUN`/`REUSE_ROWS` from Parts 0-3 (cheap; re-run them in a fresh kernel).
It writes `almac11_source_map.fits` — the per-(arm, galaxy) resolver the analysis notebook
(`analogues_specphot_almac11.ipynb`) reads, covering both new trees and reused CEERS trees.


In [ ]:
# ── Part 6 · sed_is_complete census + almac11_source_map.fits ────────────────
rows = []
for arm in ARMS:
    for key in RUN[arm]:
        snap, gid = key
        p = rtout_path_tag(arm, snap, gid)                     # new tree
        rows.append((arm, snap, gid, arm, p, sed_is_complete(p)))
for r_ in REUSE_ROWS:
    if r_[5]:                                                  # only complete reused SEDs
        rows.append(r_)

S = Table(rows=rows, names=('ARM', 'SNAPSHOT', 'GROUPID_SNAPSHOT', 'TREE_TAG',
                            'RTOUT_PATH', 'COMPLETE'))
S.write(os.path.join(OUTDIR, 'almac11_source_map.fits'), overwrite=True)

print(f'{"arm":12s} {"planned":>8s} {"complete":>9s}')
for arm in ARMS:
    m_ = np.asarray(S['ARM']) == arm
    print(f'{arm:12s} {int(m_.sum()):8d} {int(np.asarray(S["COMPLETE"])[m_].sum()):9d}')
print(f"\nsource map -> {os.path.join(OUTDIR, 'almac11_source_map.fits')}")
print('re-run this cell as the RT progresses; the analysis notebook only reads COMPLETE rows.')
